# Simple Disaster COG Processing → nasa-disasters-staging

One pass: **non-COG source → COG with activation tags → final location.** There is no
`drcs_activations_new` staging hop and no follow-up notebook.

- **Source:** `s3://nasa-disasters/drcs_activations/<EVENT>/<SUB_PRODUCT>/` — raw, non-COG GeoTIFFs.
- **Destination:** `s3://nasa-disasters-staging/ProgramData/<PRODUCT>/Output/` — the final location.
- **The activation event lives in the GeoTIFF tags, not the filename.** `ACTIVATION_METADATA` is
  embedded at COG-creation time (`ACTIVATION_EVENT` plus the derived `YEAR_MONTH` / `HAZARD` /
  `LOCATION`, and `SOURCE` / `PROCESSOR`). Output names carry no `YYYYMM_Hazard_Location_` prefix.

> Compare with [`simple_disaster_template.ipynb`](simple_disaster_template.ipynb), which writes to
> `nasa-disasters/drcs_activations_new/` **with** the event prefixed onto the filename, and
> [`bake_event_metadata.ipynb`](bake_event_metadata.ipynb), which is the **backfill** tool for COGs
> that are already on S3 without tags. Neither is needed after a run of this notebook.

## Available CLI Tools
- `aws s3 ls / cp` - List and transfer S3 files
- `rio cogeo validate` - Validate COG structure
- `gdalinfo` - Inspect GeoTIFF metadata and tags

---

## Step 1: Configuration

Set your event details and S3 paths:

In [ ]:
# ========================================
# INPUTS
# ========================================

# ---- Source: raw (non-COG) GeoTIFFs staged for the activation ----
SOURCE_BUCKET = 'nasa-disasters'             # S3 bucket holding the raw delivery
GEOTIFF_DIR = 'drcs_activations'             # Where non-converted source files live

# ---- Event Details ----
EVENT_NAME = '202606_Earthquake_Venezuela'   # YYYYMM_Hazard_Location
SOURCE = "Vantor"                            # Data origin (e.g., USGS, Copernicus, CSDA, Vantor, TBD)
SUB_PRODUCT_NAME = 'Vantor'                  # Sub-directory under the event
SOURCE_PATH = f'{GEOTIFF_DIR}/{EVENT_NAME}/{SUB_PRODUCT_NAME}'

# ---- Destination: the FINAL location. No intermediate hop. ----
# {product} is filled in per category from OUTPUT_DIRS (Step 3), so NISAR lands in
# ProgramData/NISAR/Output/. Replace {product} with a literal to flatten every
# category into one folder.
DESTINATION_BUCKET = 'nasa-disasters-staging'
DESTINATION_TEMPLATE = 'ProgramData/{product}/Output'

# The activation event is carried by ACTIVATION_METADATA (GeoTIFF tags), NOT by the
# filename. '' makes prefix_event() a no-op, so every builder in FILENAME_CREATORS
# emits an unprefixed name. Set this to EVENT_NAME to restore the older
# <EVENT>_<stem>_<date>_day.tif convention.
NAME_EVENT_PREFIX = ''

# ---- Processing Options ----
OVERWRITE = False           # True to replace existing files
VERIFY = True               # True to re-read one uploaded COG and check its tags (Step 7)
COMPRESSION = 'ZSTD'        # COG compression codec
COMPRESSION_LEVEL = 9       # ZSTD level: 1 = fast/larger … 22 = slow/smallest; 9 is a balanced default

# None preserves source CRS (fastest). Uncomment EPSG:3857 for veda-data-airflow build_stac.
TARGET_CRS = None
# TARGET_CRS = "EPSG:3857"

print(f"Source:      s3://{SOURCE_BUCKET}/{SOURCE_PATH}")
print(f"Destination: s3://{DESTINATION_BUCKET}/{DESTINATION_TEMPLATE.format(product='<category>')}/")
print(f"Event in filename: {NAME_EVENT_PREFIX or '(no — tags only)'}")

In [ ]:
from shared_utils import PROCESSOR_STRING
ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
    # Add any custom key-value pairs here
}


## Step 2: List S3 Files

See what files are available before processing:

In [ ]:
import subprocess, os

# List .tif / .TIF files in the source path.
# aws-cli returns the real S3 keys (CSDA ships uppercase .TIF). We lowercase the
# key ONLY for the extension test — `key` itself keeps its true case so the
# later `aws s3 cp` still resolves the real object on S3.
result = subprocess.run(
    ['aws', 's3', 'ls', f's3://{SOURCE_BUCKET}/{SOURCE_PATH}/', '--recursive'],
    capture_output=True, text=True
)

files = []
if result.returncode == 0:
    for line in result.stdout.strip().split('\n'):
        if not line.strip():
            continue
        parts = line.strip().split()
        key = parts[-1]
        if not key.lower().endswith('.tif'):   # matches .tif / .TIF / .Tif — any case
            continue
        size_bytes = int(parts[2])
        size_gb = size_bytes / (1024**3)
        filename = os.path.basename(key)
        files.append(key)
        print(f"  {filename:<60} ({size_gb:.2f} GB)")
    print(f"\nFound {len(files)} .tif/.TIF files")
else:
    print(f"Error listing files: {result.stderr}")

## Step 3: Define Filename Transformations

Based on the files listed above, configure how filenames should be renamed and where each category goes:

In [ ]:
# ========================================
# CATEGORIZATION AND OUTPUT CONFIGURATION
# ========================================
# Pre-wired for the common NASA Disasters product suite. Delete categories you
# don't need for this event, or add a new one by giving it three things: a regex
# in CATEGORIZATION_PATTERNS, a subdir in OUTPUT_DIRS, and (optionally) a value
# in NODATA_VALUES. The filename builder is shared by every category.

import os
import re

# STEP 1: Filename builder. This is the shared library's single source of truth
# (shared_utils.file_naming.create_output_filename -- see CLAUDE.md "One
# filename module"); every category uses it rather than a hand-rolled copy.
#
# It is IDEMPOTENT: a source that already carries the event prefix and/or a
# trailing datetime stamp is left alone instead of accumulating a second copy
# of each. The hand-rolled builders this replaced looked only for a bare
# YYYYMMDD run, so an already-stamped vendor name fell through to their
# "no date found" branch and came out doubled:
#
#   202607_Fire_OR_SkySat_SR_TrueColor_2026-08-12T123721Z.tif
#     -> 202607_Fire_OR_202607_Fire_OR_SkySat_..._2026-08-12T123721Z_day.tif
#
# What it produces now:
#
#   SkySat_SR_TrueColor_20260812.tif
#     -> <EVENT>_SkySat_SR_TrueColor_2026-08-12_day.tif
#   202607_Fire_OR_SkySat_SR_TrueColor_2026-08-12T123721Z.tif
#     -> unchanged (already prefixed, already stamped)
#
# To give ONE product a different convention, define a function taking
# (original_path, event_name) and point that category at it in
# FILENAME_CREATORS below. `create_nisar_filename` is the worked example.
from shared_utils.file_naming import create_output_filename, create_nisar_filename

# STEP 2: Categorization patterns (regex -> category). Files matching no
# pattern are skipped with a warning. FIRST match wins, so order matters:
# 'nisar' is first because an interferogram name must never be siphoned into a
# generic optical category by a loose substring pattern.
CATEGORIZATION_PATTERNS = {
    'nisar': r'NISAR|GUNW',
    'trueColor': r'trueColor|truecolor|true_color|(?:^|[_-])tc(?=[_.-]|$)|trueColorMasked',
    'colorInfrared': r'colorInfrared|colorIR|color_infrared|CIR',
    'naturalColor': r'naturalColor|natural_color|natural',
    'shortwaveIR': r'shortwaveInfrared|swir',
    'ndvi': r'NDVI|ndvi',
    'mndwi': r'mNDWI|mndwi|MNDWI',
    'ndwi': r'NDWI|ndwi',
    'waterExtent': r'WaterExtent|waterextent|water_extent',
    'blackmarble': r'BlackMarble',
    'cloudMask': r'cloudMask|cloudMasks|cloud_mask|cloud_masks',
    'dnbr': r'dnbr|DNBR|dNBR',
    'nbr': r'nbr|NBR',
    'ssrun': r'ssrun|SSRUN',
    'psh': r'R1C1',
}

# STEP 3: category -> filename builder. All categories share the one shared-library
# convention; override a single product by assigning your own function here
# (e.g. FILENAME_CREATORS['psh'] = my_psh_namer).
FILENAME_CREATORS = {category: create_output_filename for category in CATEGORIZATION_PATTERNS}

# NISAR override. An interferogram is derived from TWO acquisitions, so its name
# carries two dates, and create_output_filename relocates only the FIRST one it
# finds -- promoting the *reference* date to the canonical trailing slot and
# stranding the secondary date mid-name, still unhyphenated:
#
#   NISAR_D54_GUNW_20260617_20260629_unw_delon_deRamp_maskWater_cm.tif
#     -> <EVENT>_NISAR_D54_GUNW_20260629_unw_..._cm_2026-06-17_day.tif   # WRONG
#
# create_nisar_filename keeps both dates, in source order (NISAR names the
# reference first), adjacent, immediately before the _day suffix -- so the name
# still ends in a date + granularity suffix, and anything reading the LAST date
# token gets the secondary (post-event) acquisition:
#
#     -> <EVENT>_NISAR_D54_GUNW_unw_delon_deRamp_maskWater_cm_2026-06-17_2026-06-29_day.tif
#
# It falls back to create_output_filename when a stem holds fewer than two
# dates, so it is safe for the whole category.
FILENAME_CREATORS['nisar'] = create_nisar_filename

# STEP 4: category -> output subdirectory, RELATIVE to the S3 destination base
# (DESTINATION_BASE is prepended when the dest key is built). Keep these as plain
# subdirs (no leading base var) so an empty value can't produce a "//" key.
OUTPUT_DIRS = {
    'nisar': 'NISAR',
    'trueColor': 'trueColor',
    'colorInfrared': 'colorIR',
    'naturalColor': 'naturalColor',
    'shortwaveIR': 'shortwaveIR',
    'ndvi': 'NDVI',
    'mndwi': 'MNDWI',
    'ndwi': 'NDWI',
    'waterExtent': 'waterExtent',
    'blackmarble': 'BlackMarble',
    'cloudMask': 'cloudMask',
    'nbr': 'NBR',
    'dnbr': 'dNBR',
    'ssrun': 'SSRUN',
    'psh': 'PSH',
}

# OPTIONAL: category -> nodata value (None / missing = auto-detect from dtype).
#
# 'nisar' is deliberately None -- VERIFY IT before a real run (see the probe
# cell below). NISAR unwrapped products are float displacement in cm, where
# 0 is a LEGITIMATE sample (zero displacement), exactly like 0 dB on the SAR
# sensors -- so 0 must never be the sentinel here. None means "inherit the
# source's own nodata tag, else auto-detect from dtype"; if these files carry
# no tag, auto-detect declares -9999.0 for float, which is wrong if the actual
# fill is NaN. Set the real value once you have read it off the source.
NODATA_VALUES = {
    'nisar': None,
    'trueColor': 0,
    'colorInfrared': 0,
    'naturalColor': 0,
    'shortwaveIR': 0,
    'ndvi': -9999,
    'mndwi': -9999,
    'ndwi': -9999,
    'waterExtent': -9999,
    'blackmarble': -9999,
    'cloudMask': 0,
    'nbr': -9999,
    'dnbr': -9999,
    'ssrun': 9.9990003E+20,
    'psh': 1,
}


In [ ]:
# ========================================
# OPTIONAL: probe the source nodata / dtype before committing to NODATA_VALUES
# ========================================
# Reads only the GeoTIFF header over /vsis3 (no download). Run this whenever a
# category's nodata is set to None and you want the real value rather than
# convert_to_cog's dtype auto-detect (uint8 -> 0, int16/float -> -9999).
#
# What to do with the answer, for a float product like NISAR displacement:
#   "NoData Value=nan"    -> the tag exists; leave None (it is inherited).
#   "NoData Value=-9999"  -> leave None (inherited), or set -9999 to be explicit.
#   no NoData line at all -> auto-detect will DECLARE -9999.0. If the actual
#                            fill is NaN, that is wrong: set the sentinel that
#                            matches the pixels, or pass False to declare none.
# Never set 0 for a displacement/backscatter product -- 0 is real data.

PROBE_CATEGORY = 'nisar'   # set to None to skip this cell

if PROBE_CATEGORY:
    probe_keys = [
        k for k in files
        if re.search(CATEGORIZATION_PATTERNS[PROBE_CATEGORY], os.path.basename(k), re.IGNORECASE)
    ]
    if not probe_keys:
        print(f"No files matched category '{PROBE_CATEGORY}' — nothing to probe.")
    else:
        key = probe_keys[0]
        print(f"Probing s3://{SOURCE_BUCKET}/{key}\n")
        info = subprocess.run(
            ['gdalinfo', f'/vsis3/{SOURCE_BUCKET}/{key}'],
            capture_output=True, text=True,
        )
        if info.returncode != 0:
            print(f"gdalinfo failed (check AWS credentials / GDAL install):\n{info.stderr}")
        else:
            lines = [
                ln.strip() for ln in info.stdout.splitlines()
                if re.search(r'NoData|Type=|Band \d+|Size is|Coordinate System|ID\["EPSG"', ln)
            ]
            for ln in lines:
                print(f"  {ln}")
            if not any('NoData' in ln for ln in lines):
                print("\n  !! No NoData tag on the source. convert_to_cog will "
                      "auto-detect from dtype — confirm that matches the actual fill.")
            print(f"\n  Current NODATA_VALUES['{PROBE_CATEGORY}'] = "
                  f"{NODATA_VALUES.get(PROBE_CATEGORY)!r}")


## Step 4: Preview All Transformations

In [ ]:
# Build the processing plan: categorize each file, then look up its filename
# builder, output subdir, and nodata value from the config above. Files that
# match no CATEGORIZATION_PATTERNS entry are skipped (add a pattern to include).
#
# Two things keep the event out of the output name:
#   - strip_event_prefix() removes one the SOURCE already carries. EVENT_NAME wins
#     (case-insensitively, and it is the only way to strip a location token that
#     itself contains underscores); otherwise any YYYYMM_Hazard_Location_ shape is
#     removed, so a misnamed delivery is still cleaned.
#   - the builders are called with NAME_EVENT_PREFIX (default ''), so no new prefix
#     is added.
# The event lives in the COG's tags and in the S3 prefix instead.
from shared_utils.file_naming import strip_event_prefix

processing_plan = []
skipped_uncat = []
renamed_from_event = []

for s3_key in files:
    filename = os.path.basename(s3_key)

    category = None
    for cat, pattern in CATEGORIZATION_PATTERNS.items():
        if re.search(pattern, filename, re.IGNORECASE):
            category = cat
            break

    if category is None:
        skipped_uncat.append(filename)
        continue

    source_stem = strip_event_prefix(filename, EVENT_NAME)
    if source_stem != filename:
        renamed_from_event.append((filename, source_stem))

    new_name = FILENAME_CREATORS[category](source_stem, NAME_EVENT_PREFIX)
    output_dir = OUTPUT_DIRS[category]
    nodata = NODATA_VALUES.get(category)  # None -> convert_to_cog auto-detects
    dest_key = f"{DESTINATION_TEMPLATE.format(product=output_dir)}/{new_name}"

    processing_plan.append({
        # 'idx' namespaces this item's /tmp files in Step 5. Two sources can share
        # a basename (different folders, processed concurrently), and with
        # NAME_EVENT_PREFIX='' an already-canonical source has new_name == filename
        # — which would otherwise make convert_to_cog read and write one path.
        'idx': len(processing_plan),
        'source': s3_key,
        'dest': dest_key,
        'filename': filename,
        'new_name': new_name,
        'category': category,
        'nodata': nodata,
    })
    print(f"  {filename}  [{category}, nodata={nodata}]")
    print(f"    -> s3://{DESTINATION_BUCKET}/{dest_key}\n")

if skipped_uncat:
    print(f"Skipped {len(skipped_uncat)} uncategorized file(s) — add a pattern to "
          f"CATEGORIZATION_PATTERNS to include them:")
    for f in skipped_uncat:
        print(f"    - {f}")

if renamed_from_event:
    print(f"\nStripped an event prefix already present in {len(renamed_from_event)} source name(s):")
    for before, after in renamed_from_event:
        print(f"    {before}\n      -> {after}")

# Anything still shaped like an event prefix got past both passes -- surface it
# rather than shipping a doubled name. Usually means a location token with
# underscores that does not match EVENT_NAME.
_prefixed = [p['new_name'] for p in processing_plan
             if not NAME_EVENT_PREFIX
             and re.match(r'^\d{6}_[A-Za-z0-9]+_[A-Za-z0-9]+_', p['new_name'])]
if _prefixed:
    print(f"\n!! {len(_prefixed)} output name(s) STILL look event-prefixed — check EVENT_NAME:")
    for n in _prefixed:
        print(f"    - {n}")

print(f"\nTotal: {len(processing_plan)} files to process")

## Step 5: Process Files (Download, COG Convert, Upload)

Downloads each source file, converts it to a COG with the activation tags embedded, and uploads it
straight to its final location in `nasa-disasters-staging`.

In [ ]:
import time
from botocore.exceptions import ClientError
from shared_utils.parallel import map_threaded
from shared_utils.s3_operations import initialize_s3_client, upload_to_s3

# ONE client for every worker. initialize_s3_client tries the STS assume-role from
# aws_credentials.py first — that is what grants write on nasa-disasters-staging —
# and falls back to ambient credentials, printing which one it used. Read that line
# before blaming an AccessDenied on the bucket policy.
# Built once on purpose: boto3 clients are thread-safe for head_object/upload_file,
# and a per-worker client would repeat the STS call four times.
s3_client, _ = initialize_s3_client(DESTINATION_BUCKET)
if s3_client is None:
    raise RuntimeError(f"Could not initialize an S3 client for {DESTINATION_BUCKET}")


def _exists(key):
    try:
        s3_client.head_object(Bucket=DESTINATION_BUCKET, Key=key)
        return True
    except ClientError as exc:
        code = exc.response.get('Error', {}).get('Code')
        if code in ('404', 'NoSuchKey'):
            return False
        # e.g. 403 when the role can write but not read. Don't silently claim the
        # object is absent — say so, then let the upload proceed.
        print(f"  WARNING: could not check {key} ({code}); assuming it does not exist")
        return False


# Pre-skip items that already exist so worker threads don't waste effort.
pending = []
results = []
for item in processing_plan:
    if not OVERWRITE and _exists(item['dest']):
        print(f"SKIPPED (exists): {item['filename']}")
        results.append({'file': item['filename'], 'status': 'skipped', 'time': 0})
        continue
    pending.append(item)


def _process(item):
    src_key = item['source']
    dest_key = item['dest']
    filename = item['filename']
    new_name = item['new_name']
    nodata = item.get('nodata')  # per-category; None -> convert_to_cog auto-detects
    start = time.time()
    # Namespaced by the plan index: `new_name` can equal `filename` here (nothing
    # adds an event prefix any more), and two sources in different folders can
    # share a basename across the four worker threads.
    local_input = f"/tmp/{item['idx']}_src_{filename}"
    local_cog = f"/tmp/{item['idx']}_cog_{new_name}"
    try:
        # 1. Download the raw source (ambient read on nasa-disasters)
        subprocess.run(
            ['aws', 's3', 'cp', f's3://{SOURCE_BUCKET}/{src_key}', local_input],
            capture_output=True, text=True, check=True,
        )
        # 2. Convert to COG (reprojects + embeds the activation tags + clips if needed)
        from shared_utils.cog_utils import convert_to_cog
        cog_path = convert_to_cog(
            local_input,
            output_cog=local_cog,
            nodata=nodata,
            dst_crs=TARGET_CRS,
            metadata=ACTIVATION_METADATA,
            compression=COMPRESSION,
            compression_level=COMPRESSION_LEVEL,
        )
        # 3. Upload to the final location (assume-role client; multipart above 100 MB)
        if not upload_to_s3(s3_client, cog_path, DESTINATION_BUCKET, dest_key, verbose=False):
            raise RuntimeError(f"upload failed: s3://{DESTINATION_BUCKET}/{dest_key}")
        return {'file': filename, 'status': 'success', 'time': time.time() - start}
    except Exception as e:
        return {'file': filename, 'status': 'failed', 'time': time.time() - start, 'error': str(e)}
    finally:
        for tmp in (local_input, local_cog):
            if os.path.exists(tmp):
                try:
                    os.remove(tmp)
                except OSError:
                    pass


# max_workers=4: S3 I/O + GDAL native both release the GIL, so threads overlap.
processed = map_threaded(_process, pending, max_workers=4, desc="Simple disaster batch")
for item, out in zip(pending, processed):
    if isinstance(out, Exception):
        results.append({'file': item['filename'], 'status': 'failed', 'time': 0, 'error': str(out)})
    else:
        results.append(out)
        print(f"  {out['status'].upper()}: {out['file']} ({out['time']:.1f}s)")

print(f"\nProcessed {sum(1 for r in results if r['status']=='success')}/{len(results)} successfully")

## Step 6: Results Summary

In [ ]:
total = len(results)
success = sum(1 for r in results if 'success' in r['status'])
failed = sum(1 for r in results if r['status'] == 'failed')
skipped = sum(1 for r in results if r['status'] == 'skipped')

print(f"Total files:  {total}")
print(f"  Success:    {success}")
print(f"  Failed:     {failed}")
print(f"  Skipped:    {skipped}")

if success > 0:
    times = [r['time'] for r in results if 'success' in r['status']]
    print(f"\nAvg time: {sum(times)/len(times):.1f}s per file")

if failed > 0:
    print("\nFailed files:")
    for r in results:
        if r['status'] == 'failed':
            print(f"  - {r['file']}: {r.get('error', 'Unknown')}")

## Step 7: Verify the activation tags survived

The event is no longer in the filename, so these tags are the **only** record of which activation a
COG belongs to. Pull one finished object back out of `nasa-disasters-staging` and confirm.

In [ ]:
# Re-read one uploaded COG and check the activation tags + COG structure.
REQUIRED_TAGS = ('ACTIVATION_EVENT', 'YEAR_MONTH', 'HAZARD', 'LOCATION', 'SOURCE', 'PROCESSOR')

if not VERIFY:
    print("VERIFY = False — skipping.")
else:
    ok = [r for r in results if r['status'] == 'success']
    if not ok:
        print("No successful uploads to verify.")
    else:
        sample = next(i for i in processing_plan if i['filename'] == ok[0]['file'])
        local_verify = f"/tmp/verify_{sample['idx']}_{sample['new_name']}"
        print(f"Verifying s3://{DESTINATION_BUCKET}/{sample['dest']}\n")
        try:
            s3_client.download_file(DESTINATION_BUCKET, sample['dest'], local_verify)

            info = subprocess.run(['gdalinfo', local_verify], capture_output=True, text=True)
            in_meta = False
            for line in info.stdout.splitlines():
                if line.startswith('Metadata:'):
                    in_meta = True
                    print(line)
                elif in_meta:
                    if line.startswith('  '):
                        print(line)
                    else:
                        break

            missing = [t for t in REQUIRED_TAGS if f"{t}=" not in info.stdout]
            if missing:
                print(f"\n  !! MISSING TAGS: {missing} — the event is now unrecoverable from this "
                      f"COG. Check that ACTIVATION_METADATA was passed to convert_to_cog.")
            else:
                print(f"\n  All activation tags present: {list(REQUIRED_TAGS)}")

            val = subprocess.run(['rio', 'cogeo', 'validate', local_verify],
                                 capture_output=True, text=True)
            print(f"\n{val.stdout or val.stderr}")
        finally:
            if os.path.exists(local_verify):
                os.remove(local_verify)

## Troubleshooting

- **"No files found"** - Check `SOURCE_PATH`, verify bucket permissions, ensure `.tif` extension
- **Files being skipped** - Set `OVERWRITE = True` in Step 1
- **`AccessDenied` on upload** - `initialize_s3_client` prints which credential method it used. If it
  says it fell back to default credentials, the STS assume-role config (`aws_credentials.py`, which is
  git-ignored) is missing or not importable, and your ambient role probably can't write
  `nasa-disasters-staging`.
- **`WARNING: could not check <key> (403)`** - the role can write but not read the destination. The
  skip-if-exists check degrades to "always upload"; the upload itself is unaffected.
- **Step 7 reports missing tags** - `ACTIVATION_METADATA` didn't reach `convert_to_cog`. Re-run the
  cell under Step 1 that builds the dict, then re-process. `convert_to_cog` embeds tags only through
  its in-process `rio_cogeo` path (`metadata=` set); GDAL 3.10+ refuses to add them to a finished COG
  afterwards.
- **Event name shows up in the output filename** - `NAME_EVENT_PREFIX` is not `''`.
- **Processing errors** - Check `aws configure` has valid credentials, verify disk space in `/tmp`
- **Wrong CRS** - Inspect with `!gdalinfo /tmp/yourfile.tif` and adjust `TARGET_CRS`